# Учись Машина Учись / Learn Machine Learn

<img src="data/lml.png" width=200>

Онлайн-лекции Ильи С. Елисеева: применение методов машинного обучения в анализе данных.

- Канал в Telegram: https://t.me/learn_machine_learn
- YouTube: https://www.youtube.com/channel/UCCwDwKatNitBCVAJajremMQ
- VK: https://vk.com/learn_machine_learn
- GitHub: https://github.com/easyise/learn_machine_learn

---



# Почему Random Forest - отстой?

Почему это не самая лучшая модель для всех задач?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## 1. Случайность "леса"

В основе "Случайного леса" лежит ансамблевый метод "беггинг", который строит множество деревьев решений на случайных подвыборках данных и объединяет их предсказания для улучшения общей точности модели.

Рассмотрим на примере задачи распознавания цифр из набора данных `load_digits`.

In [ ]:
X, y = load_digits(return_X_y=True)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

X_train.shape, X_val.shape


In [ ]:
X[0]

In [ ]:
X[0].reshape(8, 8)

In [ ]:
plt.imshow(X[1].reshape(8, 8), cmap='gray_r')
plt.show()

In [ ]:
seed = 20261029
# seed = 42

model = RandomForestClassifier(
        n_estimators=20,
        max_depth=7,
        random_state=seed
    )

model.fit(X_train, y_train)
pred = model.predict(X_val)
print("Seed: %s, Accuracy: %s%%" % (seed, round(accuracy_score(y_val, pred), 4) * 100 ))

In [ ]:
import random
random.seed(10420261029)

scores = []

for _ in range(10):
    seed = int(random.random() * 1e6)
    model = RandomForestClassifier(
        n_estimators=20,
        max_depth=7,
        random_state=seed
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    score = round(accuracy_score(y_val, pred), 4) * 100
    scores.append(score)

plt.title(f"Random Forest, {X.shape[0]} digits")
plt.xlabel("Iteration")
plt.ylabel("Accuracy, %")
plt.plot(scores);

## 2. Интерпретируемость "леса"

Следствие его "случайности". Нельзя 100% полагаться на `feature_importances_` для интерпретации важности признаков.


Рассмотрим на примере датасета "Качество вина".

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
wine_data = pd.read_csv(url, sep=";")

X = wine_data.drop('quality', axis=1)
y = wine_data['quality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(X_train.shape, X_test.shape)
wine_data

In [ ]:
wine_data['quality'].value_counts()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

global_importances = pd.Series(
    rf.feature_importances_, # Feature importances from the trained Random Forest model
    index=X_train.columns)
global_importances.sort_values(ascending=True, inplace=True)
global_importances.plot.barh(color='green')
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.show()

In [ ]:
X_random = X.copy()
RANDOM_NOISE = np.random.RandomState(42).rand(X.shape[0])
RANDOM_NNOISE = np.random.RandomState(42).randn(X.shape[0])
RANDOM_CAT = np.random.RandomState(42).choice([0,1,2,3,4], size=X.shape[0])

plt.hist(RANDOM_NOISE, bins=100, color='blue', alpha=0.7)
plt.show()
# plt.hist(RANDOM_NNOISE, bins=100, color='green', alpha=0.7)
# plt.show()
# plt.hist(RANDOM_CAT, bins=100, color='red', alpha=0.7)
# plt.show()

In [ ]:
from ipywidgets import interact, widgets

X_random["RANDOM"] = RANDOM_NOISE
# X_random["RANDOM"] = RANDOM_NNOISE
# X_random["RANDOM"] = RANDOM_CAT
X_train, X_test, y_train, y_test = train_test_split(X_random, y, test_size=0.3, random_state=42)
print(X_train.shape, X_test.shape)

def plot_feature_importance(imp):
    seed = np.random.default_rng(imp).integers(2023042100, 2023042200)

    rf_random = RandomForestRegressor(n_estimators=42, random_state=seed)
    rf_random.fit(X_train, y_train)

    imp_rand = pd.Series(rf_random.feature_importances_, index=X_train.columns)
    imp_rand.sort_values(inplace=True)

    fig, ax = plt.subplots()
    imp_rand.plot.barh(
        ax=ax,
        color=['red' if x.startswith('RANDOM') else 'green' for x in imp_rand.index],
    )
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    ax.set_title(f"Random Forest Feature Importance ({seed})")
    plt.show()

interact(plot_feature_importance, imp=widgets.IntSlider(0, 0, 10, continuous_update=False))

In [ ]:
rf_random = RandomForestRegressor(n_estimators=42, random_state=2023042171)
rf_random.fit(X_train, y_train)

imp_rand = pd.Series(rf_random.feature_importances_, index=X_train.columns)
imp_rand.sort_values(ascending=True, inplace=True)
imp_rand.plot.barh(color=list(map(lambda x: 'red' if x.startswith('RANDOM') else 'green', 
                                imp_rand.index)))
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(f"Random Forest Feature Importance")
plt.show()

In [ ]:
features_to_keep = imp_rand[imp_rand > imp_rand.loc["RANDOM"]]
features_to_keep = features_to_keep.index.tolist()
features_to_keep

In [ ]:
X_train_reduced = X_train.loc[:, features_to_keep]
X_train_reduced

In [ ]:
X_test_reduced = X_test.loc[:, features_to_keep]
X_test_reduced

In [ ]:
from sklearn.metrics import root_mean_squared_error

rf = RandomForestRegressor(n_estimators=42, random_state=2023042171)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

rf_reduced = RandomForestRegressor(n_estimators=42, random_state=2023042171)
rf_reduced.fit(X_train_reduced, y_train)
y_pred_reduced = rf_reduced.predict(X_test_reduced)

print("RMSE на полных данных и на сокращенных данных:")
print("Полные данные:", root_mean_squared_error(y_test, y_pred))
print("Сокращенные данные:", root_mean_squared_error(y_test, y_pred_reduced))


In [ ]:
from sklearn.inspection import permutation_importance

perm_importance = permutation_importance(rf_random, X_test, y_test, 
    scoring="neg_mean_squared_error",
    n_repeats=20,
    n_jobs=-1,
    random_state=20261029)

perm = pd.Series(
    perm_importance.importances_mean,
    index=X_test.columns
)

perm.sort_values(ascending=True, inplace=True)

perm.plot.barh(
    ax=plt.gca(), title="Permutation importance Random Forest", color=list(map(lambda x: 'red' if x.startswith('RANDOM') else 'green', 
                                perm.index))
)

plt.tight_layout()
plt.show()

In [ ]:
perm

In [ ]:
from xgboost import XGBRegressor

model_xgb = XGBRegressor(n_estimators=150, random_state=42)
model_xgb.fit(X_train, y_train)

imp_rand = pd.Series(model_xgb.feature_importances_, index=X_train.columns)
imp_rand.sort_values(ascending=True, inplace=True)
imp_rand.plot.barh(color=list(map(lambda x: 'red' if x.startswith('RANDOM') else 'green', 
                                  imp_rand.index)))
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("XGBoost Feature Importance")
plt.show()

In [ ]:
perm_importance = permutation_importance(model_xgb, X_test, y_test, 
    scoring="neg_mean_squared_error",
    n_repeats=20,
    n_jobs=-1,
    random_state=20261029)

perm = pd.Series(
    perm_importance.importances_mean,
    index=X_test.columns
)

perm.sort_values(ascending=True, inplace=True)

perm.plot.barh(
    ax=plt.gca(), title="Permutation importance XGBoost", color=list(map(lambda x: 'red' if x.startswith('RANDOM') else 'green', 
                                perm.index))
)

plt.tight_layout()
plt.show()